In [1]:
import pandas as pd
insurance = pd.read_csv("insurance.csv")
insurance.head()

FileNotFoundError: [Errno 2] No such file or directory: 'insurance.csv'

In [ ]:
insurance.info()

In [ ]:
insurance.describe(include="all")

In [ ]:
missing_values = insurance.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
less = missing_values[missing_values < 1000].index
over = missing_values[missing_values >= 1000].index
print(f"Kolom aman: {less.tolist()}")
print(f"Kolom banyak hilang: {over.tolist()}")

In [ ]:
# Contoh mengisi nilai yang hilang dengan median untuk kolom numerik
numeric_features = insurance.select_dtypes(include='number').columns
insurance[numeric_features] = insurance[numeric_features].fillna(insurance[numeric_features].median())

In [ ]:
# Contoh mengisi nilai yang hilang dengan mode untuk kolom kategori
kategorical_features = insurance.select_dtypes(include='object').columns
for column in kategorical_features:
    insurance[column] = insurance[column].fillna(insurance[column].mode()[0])

In [ ]:
# Menghapus kolom dengan terlalu banyak nilai yang hilang
df = insurance.drop(columns=over)

In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for feature in ['age', 'bmi', 'charges']:
    plt.figure(figsize=(10, 2))
    sns.boxplot(x=df[feature])
    plt.title(f'Box Plot of {feature}')
    plt.show()

In [ ]:
#contoh sederhana untuk mengidentifikasi outliers menggunakan IQR
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

In [ ]:
# Filter dataframe untuk hanya menyimpan baris yang tidak mengandung outliers pada kolom numerik
condition = ~((df[numeric_features] < (Q1 - 1.5 * IQR)) | (df[numeric_features] > (Q3 + 1.5 * IQR))).any(axis=1)
df_filtered_numeric = df.loc[condition, numeric_features]

# Gabung kembali dengan kategorikal
df = pd.concat([df_filtered_numeric, df.loc[condition, kategorical_features]], axis=1)
print(f"Data sebelum filter: 1338, setelah filter outlier: {len(df)}")
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisasi fitur numerik
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])
print("Setelah StandardScaler:")
print(df.head())

In [ ]:
#Mengidentifikasi baris duplikat
duplicates = df.duplicated()

print("Baris duplikat:")
print(df[duplicates])

In [ ]:
#Menghapus Baris duplikat
df = df.drop_duplicates()

#cek ukuran dataframe setelah duplikatt dihapus
df.shape

In [ ]:
duplicates = df.duplicated()
print('Jumlah baris duplikat:', duplicates.sum())


In [ ]:
# Menghapus baris duplikat
df = df.drop_duplicates()
print(f'Data setelah hapus duplikat: {len(df)} baris')

In [ ]:
category_features = df.select_dtypes(include=['object']).columns
df[category_features]

In [ ]:
# One Hot Encoding
df_one_hot = pd.get_dummies(df, columns=category_features)
print(df_one_hot.head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

#Inisialisasi LabelEncoder

label_encoder = LabelEncoder()
df_lencoder = pd.DataFrame(df)

for col in category_features:
    df_lencoder[col] = label_encoder.fit_transform(df[col])

#menampilkan hasil
df_lencoder

In [ ]:
df_lencoder.head()

In [ ]:
# Menghitung jumlah variabel
num_vars = df_lencoder.shape[1]
# Menentukan grid subplot
n_cols = 4
n_rows = -(-num_vars // n_cols) # Ceiling division
# Membuat subplot
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
# Flatten
axes = axes.flatten()
# Plot setiap variabel
for i, column in enumerate(df_lencoder.columns):
    df_lencoder[column].hist(ax=axes[i], bins=20, edgecolor='black')
    axes[i].set_title(column)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
# Hapus subplot yang tidak terpakai
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
# Layout
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisasi distribusi data untuk beberapa kolom Medical Insurance
columns_to_plot = ['age', 'bmi', 'children', 'charges', 'smoker']

plt.figure(figsize=(15, 10))
for i, column in enumerate(columns_to_plot, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df_lencoder[column], kde=True, bins=30)
    plt.title(f'Distribution of {column}')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi korelasi antar variabel numerik
plt.figure(figsize=(12, 10))
correlation_matrix = df_lencoder.corr()

sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix - Medical Insurance')
plt.show()

In [ ]:
# Menghitung korelasi antara variabel target dan semua variabel lainnya
target_corr = df_lencoder.corr()['charges']

# (Opsional) Mengurutkan hasil korelasi
target_corr_sorted = target_corr.abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
target_corr_sorted.plot(kind='bar')
plt.title(f'Correlation with charges')
plt.xlabel('Variables')
plt.ylabel('Correlation Coefficient')
plt.show()

print(target_corr_sorted)

In [ ]:
# Import library SKLearn
import sklearn
from sklearn.model_selection import train_test_split

In [ ]:
# Memisahkan fitur (X) dan target (y)
X = df_lencoder.drop(columns='charges')
y = df_lencoder['charges']

print("X shape (fitur):", X.shape)
print("y shape (target):", y.shape)
print("\nKolom X:", X.columns.tolist())

In [ ]:
# 3. Membagi dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#menghitung panjang/jumlah data
print("Jumlah data: ",len(X))
#menghitung panjang/jumlah data pada x_train
print("Jumlah data latih: ",len(X_train))
#menghitung panjang/jumlah data padax_test
print("Jumlah data test: ",len(X_test))

In [ ]:
# Melatih model 1 dengan algoritma Least Angle Regression
from sklearn import linear_model
lars = linear_model.Lars(n_nonzero_coefs=1)
lars.fit(X_train, y_train)

# Melatih model 2 dengan algoritma Linear Regression
from sklearn.linear_model import LinearRegression
LR = LinearRegression()
LR.fit(X_train, y_train)

# Melatih model 3 dengan algoritma Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
GBR = GradientBoostingRegressor(random_state=184)
GBR.fit(X_train, y_train)

print("3 Model berhasil dilatih dengan Medical Insurance")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluasi pada model LARS
pred_lars = lars.predict(X_test)
mae_lars = mean_absolute_error(y_test, pred_lars)
mse_lars = mean_squared_error(y_test, pred_lars)
r2_lars = r2_score(y_test, pred_lars)

# Membuat dictionary untuk menyimpan hasil evaluasi
data = {
    'MAE': [mae_lars],
    'MSE': [mse_lars],
    'R2': [r2_lars]
}

# Konversi dictionary menjadi DataFrame
df_results = pd.DataFrame(data, index=['Lars'])
df_results

In [ ]:
# Evaluasi pada model Linear Regression
pred_LR = LR.predict(X_test)
mae_LR = mean_absolute_error(y_test, pred_LR)
mse_LR = mean_squared_error(y_test, pred_LR)
r2_LR = r2_score(y_test, pred_LR)

# Menambahkan hasil evaluasi LR ke DataFrame
df_results.loc['Linear Regression'] = [mae_LR, mse_LR, r2_LR]
df_results

In [ ]:
# Evaluasi pada model Gradient Boosting Regressor
pred_GBR = GBR.predict(X_test)
mae_GBR = mean_absolute_error(y_test, pred_GBR)
mse_GBR = mean_squared_error(y_test, pred_GBR)
r2_GBR = r2_score(y_test, pred_GBR)

# Menambahkan hasil evaluasi GBR ke DataFrame
df_results.loc['GradientBoostingRegressor'] = [mae_GBR, mse_GBR, r2_GBR]
df_results

In [ ]:
import joblib

# Menyimpan model ke dalam file
joblib.dump(GBR, 'gbr_model_medical_insurance.joblib')
print("Model disimpan sebagai gbr_model_medical_insurance.joblib")

In [ ]:
#membuat model dari file joblib
joblib_model = joblib.load('gbr_model_medical_insurance.joblib')

In [ ]:
from flask import Flask, request, jsonify
import joblib

# Inisialisasi aplikasi Flask
app = Flask(__name__)

# Memuat model yang telah disimpan
joblib_model = joblib.load('gbr_model_medical_insurance.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json['data']  # Mengambil data dari request JSON
    prediction = joblib_model.predict(data)  # Melakukan prediksi
    return jsonify({'prediction': prediction.tolist()})

if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
import joblib
import numpy as np

# load model yang udah kamu train
joblib_model = joblib.load('gbr_model_medical_insurance.joblib')

# test prediksi langsung di Jupyter
data = [[19, 0, 27.9, 0, 1, 0]]
hasil = joblib_model.predict(data)
print(hasil)

In [2]:
import pandas as pd
import os

folder = "Analisis Dataset Machine Learning"

for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(root, file)

            try:
                df = pd.read_csv(path)
                print("Nama Dataset :", file)
                print("Jumlah Baris :", df.shape[0])
                print("Jumlah Kolom :", df.shape[1])
                print("-" * 40)

            except Exception as e:
                print("Gagal membaca:", file, e)

        elif file.endswith(".xlsx"):
            path = os.path.join(root, file)

            try:
                df = pd.read_excel(path)
                print("Nama Dataset :", file)
                print("Jumlah Baris :", df.shape[0])
                print("Jumlah Kolom :", df.shape[1])
                print("-" * 40)

            except Exception as e:
                print("Gagal membaca:", file, e)

Nama Dataset : heart.csv
Jumlah Baris : 303
Jumlah Kolom : 14
----------------------------------------
Nama Dataset : o2Saturation.csv
Jumlah Baris : 3585
Jumlah Kolom : 1
----------------------------------------
Nama Dataset : predictive_maintenance.csv
Jumlah Baris : 10000
Jumlah Kolom : 10
----------------------------------------
Nama Dataset : water_potability.csv
Jumlah Baris : 3276
Jumlah Kolom : 10
----------------------------------------
Nama Dataset : Dataset.xlsx
Jumlah Baris : 128
Jumlah Kolom : 17
----------------------------------------
Nama Dataset : Dry_Bean_Dataset.xlsx
Jumlah Baris : 13611
Jumlah Kolom : 17
----------------------------------------
Nama Dataset : vehicle.csv
Jumlah Baris : 846
Jumlah Kolom : 19
----------------------------------------
Gagal membaca: Emotion.csv Error tokenizing data. C error: Expected 1 fields in line 9, saw 2

Nama Dataset : sample_submission.csv
Jumlah Baris : 8989
Jumlah Kolom : 7
----------------------------------------
Gagal membac

In [3]:
from scipy.io import arff
import pandas as pd

data, meta = arff.loadarff(
    "Analisis Dataset Machine Learning/Multilable/emotions.arff"
)

df = pd.DataFrame(data)

print("Jumlah Baris:", df.shape[0])
print("Jumlah Kolom:", df.shape[1])

Jumlah Baris: 593
Jumlah Kolom: 78
